In [22]:
import re
import nltk
import pandas as pd

from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

In [23]:
df1 = pd.read_csv('test.csv')
df2 = pd.read_csv('train.csv')

df = pd.concat([df1, df2], ignore_index = True)

In [24]:
df.columns

Index(['category', 'sub_category', 'crimeaditionalinfo'], dtype='object')

In [25]:
df.shape

(124915, 3)

In [26]:
df.head()

,category,sub_category,crimeaditionalinfo
0,RapeGang Rape RGRSexually Abusive Content,NaN,Sir namaskar mein Ranjit Kumar PatraPaise neh...
1,Online Financial Fraud,DebitCredit Card FraudSim Swap Fraud,KOTAK MAHINDRA BANK FRAUD\r\nFRAUD AMOUNT
2,Cyber Attack/ Dependent Crimes,SQL Injection,The issue actually started when I got this ema...
3,Online Financial Fraud,Fraud CallVishing,I am amit kumar from karwi chitrakoot I am tot...
4,Any Other Cyber Crime,Other,I have ordered saree and blouse from rinki s...


In [27]:
df = df.dropna(subset=["category"])
df = df.drop_duplicates(subset=["crimeaditionalinfo"])
df = df.rename(columns={"crimeaditionalinfo": "content"})
df = df.reset_index(drop=True)

In [28]:
df.head()

,category,sub_category,content
0,RapeGang Rape RGRSexually Abusive Content,NaN,Sir namaskar mein Ranjit Kumar PatraPaise neh...
1,Online Financial Fraud,DebitCredit Card FraudSim Swap Fraud,KOTAK MAHINDRA BANK FRAUD\r\nFRAUD AMOUNT
2,Cyber Attack/ Dependent Crimes,SQL Injection,The issue actually started when I got this ema...
3,Online Financial Fraud,Fraud CallVishing,I am amit kumar from karwi chitrakoot I am tot...
4,Any Other Cyber Crime,Other,I have ordered saree and blouse from rinki s...


In [29]:
nltk.download("stopwords")
nltk.download("wordnet")
nltk.download("omw-1.4")

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/triptibhardwaj/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     /Users/triptibhardwaj/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package omw-1.4 to
[nltk_data]     /Users/triptibhardwaj/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

In [30]:
stop_words = set(stopwords.words("english"))
lemmatizer = WordNetLemmatizer()

In [32]:
def preprocess_text(text):
    if pd.isna(text):
        return ""
    text = text.lower()
    text = re.sub(r"http\S+|www\S+", "", text)
    text = re.sub(r"\S+@\S+", "", text)
    text = re.sub(r"\d+", "", text)
    text = re.sub(r"[^a-z\s]", "", text)
    tokens = text.split()
    tokens = [
        lemmatizer.lemmatize(word)
        for word in tokens
        if word not in stop_words
    ]
    return " ".join(tokens)

In [33]:
df["content"] = df["content"].apply(preprocess_text)

In [34]:
df['category'].value_counts()

category
Online Financial Fraud                                  68455
Online and Social Media Related Crime                   16032
Any Other Cyber Crime                                   14393
Cyber Attack/ Dependent Crimes                           4869
Sexually Obscene material                                2347
Hacking  Damage to computercomputer system etc           2279
Sexually Explicit Act                                    1958
Cryptocurrency Crime                                      629
Online Gambling  Betting                                  572
Child Pornography CPChild Sexual Abuse Material CSAM      469
RapeGang Rape RGRSexually Abusive Content                 323
Online Cyber Trafficking                                  242
Cyber Terrorism                                           212
Ransomware                                                 74
Crime Against Women & Children                              4
Report Unlawful Content                                     1

In [35]:
category_mapping = {
    "Online Financial Fraud": "Financial Fraud",
    "Online and Social Media Related Crime": "Social Media Crime",
    "Any Other Cyber Crime": "Other Cyber Crime",
    "Cyber Attack/ Dependent Crimes": "Cyber Attack/Dependent Crimes",
    "Sexually Obscene material": "Sexually Obscene Content",
    "Hacking  Damage to computercomputer system etc": "Hacking/Damage",
    "Sexually Explicit Act": "Sexually Explicit Content",
    "Cryptocurrency Crime": "Cryptocurrency Crime",
    "Online Gambling  Betting": "Gambling/Betting",
    "Child Pornography CPChild Sexual Abuse Material CSAM": "Child Abuse Material",
    "RapeGang Rape RGRSexually Abusive Content": "Rape or Sexual Abuse Content",
    "Online Cyber Trafficking": "Cyber Trafficking",
    "Cyber Terrorism": "Cyber Terrorism",
    "Ransomware": "Ransomware",
    "Crime Against Women & Children": "REMOVE",
    "Report Unlawful Content": "REMOVE"
}

In [36]:
df["category"] = df["category"].map(category_mapping)

In [41]:
df['category'].value_counts()

category
Financial Fraud                  68455
Social Media Crime               16032
Other Cyber Crime                14393
Cyber Attack/Dependent Crimes     4869
Sexually Obscene Content          2347
Hacking/Damage                    2279
Sexually Explicit Content         1958
Cryptocurrency Crime               629
Gambling/Betting                   572
Child Abuse Material               469
Rape or Sexual Abuse Content       323
Cyber Trafficking                  242
Cyber Terrorism                    212
Ransomware                          74
Name: count, dtype: int64

In [40]:
df = df[~df["category"].isin(["REMOVE"])]

In [42]:
df = df.reset_index(drop = True)

In [43]:
df.shape

(112854, 3)

In [58]:
from sklearn.model_selection import train_test_split
df = df[["content", "category"]]

MIN_TRAIN_SAMPLES = 50

train_parts = []
test_parts = []

for cat, group in df.groupby("category"):
    if len(group) <= MIN_TRAIN_SAMPLES:
        train_parts.append(group)
    else:
        tr, te = train_test_split(
            group,
            test_size=0.2,
            random_state=42
        )
        train_parts.append(tr)
        test_parts.append(te)

train_df = pd.concat(train_parts).reset_index(drop=True)
test_df  = pd.concat(test_parts).reset_index(drop=True)

In [59]:
print("Train shape:", train_df.shape)
print("Test shape :", test_df.shape)

Train shape: (90278, 2)
Test shape : (22576, 2)


In [60]:
print("Train distribution: ", train_df["category"].value_counts())
print("Test distribution: ", test_df["category"].value_counts())

Train distribution:  category
Financial Fraud                  54764
Social Media Crime               12825
Other Cyber Crime                11514
Cyber Attack/Dependent Crimes     3895
Sexually Obscene Content          1877
Hacking/Damage                    1823
Sexually Explicit Content         1566
Cryptocurrency Crime               503
Gambling/Betting                   457
Child Abuse Material               375
Rape or Sexual Abuse Content       258
Cyber Trafficking                  193
Cyber Terrorism                    169
Ransomware                          59
Name: count, dtype: int64
Test distribution:  category
Financial Fraud                  13691
Social Media Crime                3207
Other Cyber Crime                 2879
Cyber Attack/Dependent Crimes      974
Sexually Obscene Content           470
Hacking/Damage                     456
Sexually Explicit Content          392
Cryptocurrency Crime               126
Gambling/Betting                   115
Child Abuse Materi

In [61]:
assert len(set(test_df["category"]) - set(train_df["category"])) == 0

In [64]:
train_df = train_df.rename(columns={"content": "content_processed"})
test_df = test_df.rename(columns={"content": "content_processed"})


In [66]:
train_df.columns

Index(['content_processed', 'category'], dtype='object')

In [67]:
train_df.to_csv("final_train.csv", index=False)
test_df.to_csv("final_test.csv", index=False)